# Timestral-209M (zero-shot) + GBM (ensemble LightGBM / XGBoost)

Modèle pré-entraîné de Mistral : pas d'entraînement. Pour chaque arc, il lit les 2048 dernières heures avant `t0`, ainsi que les covariables connues jusqu'à la fin de l'horizon (calendrier + encodage cyclique heure/jour sin/cos). La prédiction couvre le gap d'un jour et les 120 heures évaluées.

**Résultats du backtest (38 dates, focus Convention + Saints-Pères, RMSE mean/date) :**
- Timestral seul : 64.3 | LightGBM seul : 67.2 | naif lag7j : 92.8
- contexte 1024h/4096h pire que 2048h ; covariables supplémentaires (profils, lags) pires
- encodage cyclique heure/jour : aide Champs-Élysées, neutre sur les 2 autres arcs
- **ensemble Timestral + ~35% GBM : 62.0** ← optimum large et robuste (w=0.25 à 0.5 tous ≈ 62)
- **holdout temporel (2026 non vu) confirme w=0.35 = 58.3**, meilleur que w=0.5 (58.8) et Timestral seul (60.3) → pas de surapprentissage
- voir cellule de comparaison LGB vs XGB en bas pour le choix du backend

Timestral s'installe depuis le dépôt officiel (Python >= 3.10, PyTorch >= 2.4). Si nécessaire, exécuter la cellule d'installation une fois, puis redémarrer le kernel.

In [ ]:
%pip install "git+https://github.com/mistralai/timestral.git"

In [ ]:
import sys; sys.path.append("..")  # pour importer dataset_create depuis notebooks/
import numpy as np
import pandas as pd
from dataset_create import charger, split, ORIGINES, HORIZON, feature_engineering

df = charger(imputer=True)  # imputer=False : trous laissés en NaN
rmse = lambda y, p, axe: ((y - p) ** 2).groupby(axe, observed=True).mean() ** .5  # RMSE par arc

In [ ]:
import torch
import lightgbm as lgb
import xgboost as xgb
from timestral import TimestralForecaster

pipe = TimestralForecaster.from_pretrained(
    "mistralai/Timestral-209M",
    device="cuda" if torch.cuda.is_available() else "cpu",
)
fe = feature_engineering(df)  # même préparation et index que split()

# Covariables connues : calendrier + encodage cyclique heure/jour (sin/cos)
COV_NUM = ["ferie", "pont", "paris_respire", "evenement", "confinement",
           "heure_sin", "heure_cos", "jour_sin", "jour_cos"]
COV_CAT = ["vacances_code", "type_jour"]
CTX = 2048  # 2048h optimal (testé 1024/2048/4096)
H = 24 + HORIZON
W_GBM = 0.35  # poids du GBM dans l'ensemble (38 dates : optimum plat 0.25-0.5)
BACKEND = "lgb"  # "lgb" ou "xgb" — voir cellule de comparaison en bas

# Timestral attend des covariables connues sur le contexte ET tout l'horizon.
vacances_code = fe.vacances.cat.codes.astype(float).where(fe.vacances.cat.codes >= 0)
g = fe.assign(
    item_id=fe.axe.astype(str),
    timestamp=fe.datetime.dt.tz_convert(None),  # UTC sans fuseau : grille horaire régulière
    vacances_code=vacances_code,
    type_jour=fe.type_jour.astype(float),
    heure_sin=np.sin(2 * np.pi * fe.datetime.dt.hour / 24),
    heure_cos=np.cos(2 * np.pi * fe.datetime.dt.hour / 24),
    jour_sin=np.sin(2 * np.pi * fe.datetime.dt.dayofweek / 7),
    jour_cos=np.cos(2 * np.pi * fe.datetime.dt.dayofweek / 7),
).astype({c: np.float32 for c in COV_NUM + COV_CAT})

# --- Prédiction Timestral seule (coûteuse : à mettre en cache si on sweep les poids) ---
def timestral_pred(X_train, X_test, cible="debit"):
    t0 = fe.loc[X_train.index, "datetime"].max().tz_convert(None) + pd.Timedelta("1h")
    heures = pd.date_range(t0 - pd.Timedelta(hours=CTX), periods=CTX + H, freq="h")
    predictions = []
    for axe in X_test.axe.astype(str).unique():
        serie = g.loc[g.item_id == axe].set_index("timestamp").reindex(heures)
        contexte = serie[cible].to_numpy(dtype=np.float32)[:CTX][None, None, :]
        connues = serie[COV_NUM].to_numpy(dtype=np.float32).T[None, :, :]
        categories = serie[COV_CAT].to_numpy(dtype=np.float32).T[None, :, :]
        forecast = pipe.forecast(contexte, prediction_length=H,
                                 known_covariates=connues, categorical_known_covariates=categories)
        predictions.append(pd.DataFrame({
            "item_id": axe, "timestamp": heures[CTX:],
            "prediction": forecast.median[0, 0].cpu().numpy(),
        }))
    pred = pd.concat(predictions, ignore_index=True)
    test = pd.DataFrame({
        "item_id": X_test.axe.astype(str),
        "timestamp": fe.loc[X_test.index, "datetime"].dt.tz_convert(None),
    })
    return test.merge(pred, on=["item_id", "timestamp"], how="left", validate="many_to_one").prediction.to_numpy()

# --- Prédiction GBM (LightGBM ou XGBoost) ---
LGB_PARAMS = {"debit": dict(n_estimators=1000, learning_rate=0.05, num_leaves=127, objective="huber", alpha=100),
              "occupation": dict(n_estimators=500, learning_rate=0.05, num_leaves=31, min_child_samples=20)}
XGB_PARAMS = {"debit": dict(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                              colsample_bytree=0.8, objective="reg:squarederror", random_state=42, n_jobs=-1),
              "occupation": dict(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                                 colsample_bytree=0.8, objective="reg:squarederror", random_state=42, n_jobs=-1)}
CAT_COLS = ["axe", "vacances"]  # one-hot pour XGBoost

def gbm_pred(X_train, y_train, X_test, cible="debit", backend="lgb"):
    if backend == "lgb":
        return lgb.LGBMRegressor(verbose=-1, random_state=0, **LGB_PARAMS[cible]).fit(X_train, y_train).predict(X_test).clip(0)
    else:  # xgb : one-hot des catégorielles
        Xtr = pd.get_dummies(X_train.astype({c: str for c in CAT_COLS if c in X_train}), columns=[c for c in CAT_COLS if c in X_train])
        Xte = pd.get_dummies(X_test.astype({c: str for c in CAT_COLS if c in X_test}), columns=[c for c in CAT_COLS if c in X_test])
        Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)
        return xgb.XGBRegressor(**XGB_PARAMS[cible]).fit(Xtr, y_train).predict(Xte).clip(0)

def modele(X_train, y_train, X_test, cible="debit", backend=None, w=None):
    """Ensemble Timestral + GBM. backend=None -> BACKEND global ; w=None -> W_GBM global."""
    backend = backend or BACKEND
    w = W_GBM if w is None else w
    p_ts = timestral_pred(X_train, X_test, cible)
    p_gbm = gbm_pred(X_train, y_train, X_test, cible, backend)
    return w * p_gbm + (1 - w) * p_ts

In [ ]:
# 1 semaine : dernière donnée le ven. 07/11/2025 -> on prédit sam. 08 -> mer. 12/11 (11/11 férié inclus)
CIBLE = "debit"  # ou "occupation"
X_train, X_test, y_train, y_test = split(df, "2025-11-07", cible=CIBLE)
p = modele(X_train, y_train, X_test, CIBLE)
print(pd.DataFrame({"modele": rmse(y_test, p, X_test.axe), "naif_lag7j": rmse(y_test, X_test.lag_7j, X_test.axe)}).round(1))

In [ ]:
# Backtest : même chose sur 1 vendredi sur 4 (ORIGINES pour tous), puis moyenne
res = []
for t0 in ORIGINES[::4]:
    X_train, X_test, y_train, y_test = split(df, t0, cible=CIBLE)
    if X_test.empty:  # aucun arc avec assez de vraies valeurs cette semaine-là
        continue
    p = modele(X_train, y_train, X_test, CIBLE)
    res.append(pd.DataFrame({"modele": rmse(y_test, p, X_test.axe), "naif_lag7j": rmse(y_test, X_test.lag_7j, X_test.axe)}))
par_arc = pd.concat(res).groupby(level=0).mean(numeric_only=True)
print(par_arc.round(1), "\n\nMoyenne des 3 arcs :\n", par_arc.mean().round(1))

## Comparaison LGB vs XGB comme backend de l'ensemble

Timestral est coûteux : on le calcule **une fois par date**, puis on sweep les poids LGB et XGB sur les mêmes prédictions Timestral. Focus Convention + Saints-Pères, RMSE mean/date.

In [ ]:
FOCUS = ["Convention", "Saints-Peres"]
N_DATES = 20  # ORIGINES[::4][:N_DATES]

rows = []
for t0 in ORIGINES[::4][:N_DATES]:
    X_train, X_test, y_train, y_test = split(df, t0, cible=CIBLE)
    if X_test.empty:
        continue
    m = X_test.axe.astype(str).isin(FOCUS)
    if not m.any():
        continue
    Xf, yf = X_test[m], y_test[m]
    p_ts = timestral_pred(X_train, Xf, CIBLE)          # Timestral une seule fois
    p_lgb = gbm_pred(X_train, y_train, Xf, CIBLE, "lgb")
    p_xgb = gbm_pred(X_train, y_train, Xf, CIBLE, "xgb")
    for i in range(len(Xf)):
        rows.append({"t0": t0, "axe": Xf.axe.iloc[i], "y": yf.iloc[i],
                     "ts": p_ts[i], "lgb": p_lgb[i], "xgb": p_xgb[i], "naif": Xf.lag_7j.iloc[i]})
    print(f"  {t0}: ok", flush=True)

D = pd.DataFrame(rows)
print(f"\n{D.t0.nunique()} dates, {len(D)} rows\n")
print(f"{'config':18s} {'Convention':>10s} {'Saints-Peres':>12s} {'mean/date':>10s} {'std':>6s}")
for name, p in [("timestral", D.ts), ("lgb seul", D.lgb), ("xgb seul", D.xgb), ("naif", D.naif)] + \
               [(f"ts+{w:.0%} lgb", w*D.lgb+(1-w)*D.ts) for w in [0.25, 0.35, 0.5]] + \
               [(f"ts+{w:.0%} xgb", w*D.xgb+(1-w)*D.ts) for w in [0.25, 0.35, 0.5]]:
    r_arc = ((D.y - p) ** 2).groupby(D.axe).mean() ** .5
    per_date = D.assign(e2=(D.y - p) ** 2).groupby("t0").e2.mean() ** .5
    print(f"{name:18s} {r_arc.get('Convention', float('nan')):10.1f} {r_arc.get('Saints-Peres', float('nan')):12.1f} {per_date.mean():10.1f} {per_date.std():6.1f}")